# Device Side Logic

This section of the code is to simulate devices that subscribe to the broker. Devices must subscribe to `devices\{deviceId}\command` and `devices\{deviceId}\status\get` in order to receive commands from the application as well as status requests. In turn, the device will publish to `devices\{deviceId}\status` to report it's status if requested or in response to a command. 

With this in mind, we will design a `MQTTDeviceWrapper` to wrap the mqtt operations entirely and such that it is fully compatible with any device. This wrapper will have four public methods:
- `register_command` : This will allow devices to register command callbacks to handle commands from the application. It will take a string and a callable parameter which the wrapper will use to fill a `dict` so that received commands can fail gracefully if the user has not registered them correctly.
- `register_publisher` : Similarly, this will register the status publisher for the device so the wrapper can respond to `devices\{deviceId}\status\get`.
- `start` : To connect to the broker, using credentials stored on the device.
- `stop` : To cleanly disconnect from the broker.

The rest will be handled internally by the wrapper.

The device ID will be stored in an .env file to simulate an environment variable. Since Raspberry Pis run full linux, this will be suitable facsimile for development.

Firstly, we want to update our broker conf. We will remove the plain MQTT to just focus on TLS and include logging for detailed output.

```conf
require_certificate false

# Logging configuration
log_dest stdout
log_type error
log_type warning
log_type notice
log_type information
connection_messages true
```

We set `require_certificate` to false to disable mTLS as this is out of scope for now. The main idea is to have a working demo and I don't want to get bogged down in certificates right now. From what I understand plain TLS is a standard way of doing things.

We also we want to include device specific passwords, for the smart lock `lock-01` we will generate an easy to remember password. This has to be done offline but we will script it later for demo purposes.

```bash
docker run --rm -v "$PWD/broker:/mosquitto/config" eclipse-mosquitto \
  mosquitto_passwd -b -c /mosquitto/config/passwd lock_01 pwd01
```

Next, we have to rewrite our generate certificates script. Largely, we want to split our the CA generation from the regular `server.crt` generation. CA is the central authority that all TLS handshaking devices need, where `server.crt` is specific to each device (including the broker). These scripts can be found in the script folder and it needs to be called from the root since thats where the docker compose expects it. 

Similarly, `generate_certs.sh` is in the same folder and must be called with the right parameters which configure the CN name and directory for keys.

With our keys generated, we must rework the docker compose to mount the right directories by first ammending the `mqtt-broker` image to pull certs from its own directory and from root.

```yaml
# Mount self CA certificates
- ./broker/certs:/mosquitto/config/certs
- ./certs/ca.crt:/mosquitto/config/certs/ca.crt
```

We will also mount the broker logs and data so we can persist them to inspect after the container ends.

```yaml
      # Mount data and logs
      - mosquitto_data:/mosquitto/data
      - mosquitto_log:/mosquitto/log
...
# Persist outside of container
volumes:
  mosquitto_data:
  mosquitto_log:
```

For more debugging we will add a health check to the broker as well.

```yaml
    # Add timed healthcheck
    healthcheck:
      test: ["CMD", "mosquitto_sub", "-h", "localhost", "-p", "8883", "--cafile", "/mosquitto/config/certs/ca.crt", "-t", "$$SYS/#", "-C", "1"]
      interval: 30s
      timeout: 10s
      retries: 3
```

No we must configure the device, we will start with just one. There are only a few things we need to make this work, the first of which is an .env file which will simulate our environment variable which will be part of device - an example of the env file is below.

```.env
DEVICE_ID=lock-01
MQTT_HOST=mqtt-broker
PORT=8883

MQTT_USERNAME=lock_01
MQTT_PASSWORD=pwd01

TLS_CA_CERT=/app/certs/ca.crt

```

We can then mount that in using the `env_file` keyword.

```yaml
    env_file:
      - ./simulator/simulators_devices/lock-01/.env        # Device-specific config here
```

We also need to mount the device specific certificates and set the depenedence on the broker, this ensures the broker container is brought up first so that it has a proper chance to accept the incoming connections.

```yaml
    volumes:
      - ./simulator/simulators_devices/lock-01/certs:/certs:ro         # Mount broker’s certs directory to /certs

    depends_on:
      - mqtt-broker                       # Ensure broker starts before device tries to connect
```

Finally, we need to point it to a docker file for the device.

```yaml
      context: .                          # Lock 1 directory
      dockerfile: simulator/simulators_devices/lock-01/Dockerfile
```

That brings us nicely to the device dockerfile. Our folder structure has a simulation folder at root, from there we have two shared libraries; `mqtt_wrapper`, `base_device`. These provide the minimal working functionality as well as the entire MQTT connectivity abstracted away in a drop in module. The device base can then be used to derive to specific devices. We need to copy these as well as the project toml at root so we can install the module properly.

```Dockerfile
# Set working directory
WORKDIR /app

# Copy the app code and pyproject info
COPY simulator /app/simulator
COPY pyproject.toml /app/
COPY poetry.lock /app/

# Install poetry and use it to install project deps
RUN pip install --no-cache-dir poetry
ENV POETRY_VIRTUALENVS_CREATE=false
RUN poetry install --no-root
```

At the top of the file, we define the image we base the container of. We will use `python:3.11-slim-bookworm` as it uses an ARM based debian platform which makes it compatitble with a Rapsberry PI. There are other images that more closely mimic the Pi in terms of IO but it is heavy and slows down development.

Finally, we point the docker file to an entry point which we will define.

```Dockerfile
# Run your device
CMD ["python", "simulator/simulators_devices/lock-01/run_device.py"]

```

This brings us to the actual code, as already mentioned the MQTT stuff has been wrapped so that it completely abstracts the MQTT connectivity from the user. As long as the environment variables are set then any device in this setup (using the the topics defined in the spec) will be able to seamlesly integrate new devices with their own commands and status updates.

There is a default constructor and a `@classmethod` decorated factory function to construct from environment variable - this is the reccomended approach.

From a user perspective, there are only four methods to be concerned about.
```python
def register_command(self, action: str, handler: Callable[[], None]) -> None:
    """
    Registers a command handler for a specific action.

    Args:
        action (str): The name of the action to handle.
        handler (Callable[[], None]): A function to call when the action is received.
    """
    # Add to map
    self._command_handlers[action] = handler
    logger.info(f"[{self.device_id}] Registered handler for action '{action}'")

```

This allows the user class to register callback handlers for received commands. The action string is used to as a key to the `_command_handlers` hash map so received messages can be disected to identify the command, said command is then used to index `_command_handlers` which will return a the handler.

```python
# Check map for callback
handler = self._command_handlers.get(action)
if handler:
    logger.info(f"[{self.device_id}] Handling action: {action}")
    handler()
    # Publish status in response
    self._publish_status()
else:
    logger.warning(
        f"[{self.device_id}] No handler for action: {action}"
    )
    # Respond with missing command error
    self._publish_status(error=f"unhandled command: {action}")
```

If a command is received and the relevant handler is not registered then a warning will be logged to the device and an error will be communicated to the app.

Similarly, `register_publisher` allows user classes to register status publishers, this expects a `dict` return as it will send the output straight to broker.

```python
def register_publisher(self, publisher_fn: Callable[[], dict]) -> None:
    """
    Registers a function to publish the device's current status.

    Args:
        publisher_fn (Callable[[], dict]): Function that returns a status dictionary.
    """
    # Single status callback
    self._status_publisher = publisher_fn
    logger.info(f"[{self.device_id}] Status publisher registered")
```

If the handler is not registered or the output is not `JSON`ifable then an appropriate errors and response will be communicated.

```python
try:
    if error:
        # Error publish
        payload = json.dumps({"error": error})
    elif not self._status_publisher:
        payload = json.dumps({"error": "status callback not registered"})
    else:
        # Get status from call back
        status = self._status_publisher()
        payload = json.dumps(status)
    # Publish
    self.client.publish(self.status_topic, payload, qos=qos)
    logger.info(
        f"[{self.device_id}] Published status with QoS {qos}: {payload}"
    )
except Exception as e:
    # Fatal error, no response to app
    logger.error(f"[{self.device_id}] Failed to publish status: {e}")

```

Finally, the start and stop methods allow user classes to start and stop the MQTT client. The calls have no parameters and are self explanatory.

`DeviceBase` provides the minimal smart lock functionality, it is designed to be overidden and inherited so that other classes can bring their own flavour of functionality.

The base registers the `lock` and `unlock` commands by default. It also registers the `status` method as the publisher. `status` is empty and will raise an not implmeneted error if not overidden by derived classes - Python uses the latest binding so as long as the derived class has overridden this method at construction then this will work safely.

```python
# Register minimum commands
self.device.register_command("lock", self._lock)
self.device.register_command("unlock", self._unlock)
# Set publisher
self.device.register_publisher(self.status)
logger.info(f"[{self.device_id}] Default handlers registered.")
```

The `lock` and `unlock` methods simple set the internal state accordingly, these are called when the command is received by the MQTT wrapper.

To start the device, the `DeviceBase` has a `run` method which has a loop inside it that is interruptible by keyboard interrupt - as MVP this is suitable but might need something more sophisticated or event driven to control in future.

```python
# Start device
logger.info(f"[{self.device_id}] Starting device")
self.setup()
self.device.start()

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    logger.info(f"[{self.device_id}] Shutting down...")
    self.device.stop()
```

This brings us to `run_device.py` which is the entry point of the Dockerfile for the device. This creates a small but functional smart lock simulator that simply overides the status function and instantiates the base with it's device ID.

```python
def status(self):
    """
    Returns the current status of the smart lock.

    Returns:
        dict: A dictionary representing the device's current status.
    """
    return {
        "state": "locked" if self.state["locked"] else "unlocked",
        "battery_percent": 87,
        "firmware_version": "1.0.1"
    }
```

We then include a `__main__` clause so the file can be run as script.

```python
if __name__ == "__main__":
    """
    Entry point for running the SmartLock01 device.
    """
    try:
        SmartLock01().run()
    except Exception as ex:
        logging.getLogger("SmartLock01").exception(f"Unhandled exception in SmartLock01: {ex}")
```

This set up can be copied to include multiple devices in much the same way. To run the simulation, call the `start_sim.sh` inside VSCode.